# Jour 3 — SQL Analytique, Time Series et Visualisation · Notebook Formateur
## Formation Beobank · SAS → Python · Orsys

**Usage :** notebook formateur — code commenté, à projeter avec les slides Jour 3.
**Données :** `../data/` — même 5 tables que les jours 1 et 2.

📎 **Comparaisons SAS ↔ Python détaillées :** voir `../sas_vs_python/SAS_vs_Python.ipynb`.

⚠️ **Note données :** `TXN_X_CTR.csv` ne contient pas de montant ni de date de mouvement exploitable (colonnes absentes du fichier source). Les colonnes `MNT_MVT` et `DAT_MVT` utilisées à partir d'ici sont **simulées** (graine fixe, reproductibles) — voir la cellule Setup.

## Setup — Chargement + SQLite

In [ ]:
# ── Imports et chargement ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from pandas.tseries.offsets import DateOffset

DATA   = Path("../data")
PARAMS = dict(sep=";", na_values=".", encoding="utf-8")

ctr     = pd.read_csv(DATA / "CTR.csv",       **PARAMS)
tie     = pd.read_csv(DATA / "TIE.csv",       **PARAMS)
tie_adr = pd.read_csv(DATA / "TIE_ADR.csv",   **PARAMS)
txc     = pd.read_csv(DATA / "TIE_X_CTR.csv", **PARAMS)
txn     = pd.read_csv(DATA / "TXN_X_CTR.csv", **PARAMS)

# ── Données simulées pour l'exercice ──────────────────────────────────────
# TXN_X_CTR.csv ne fournit ni montant ni date de mouvement exploitable
# (uniquement des libellés). On simule ces deux colonnes pour les besoins
# pédagogiques : montant ~ loi normale, date = date de création du mouvement.
rng = np.random.default_rng(42)                          # graine fixe → résultats reproductibles
txn["DAT_MVT"] = pd.to_datetime(txn["DAT_CRE_MVT_CPB"])   # date de mouvement simulée
txn["MNT_MVT"] = rng.normal(250, 400, size=len(txn)).round(2)  # montant simulé (moyenne 250€, écart-type 400€)

# Charger dans SQLite — permet d'exécuter du SQL pur sur les données
conn = sqlite3.connect(":memory:")
for nom, df in [("CTR",ctr),("TIE",tie),("TIE_ADR",tie_adr),
                ("TIE_X_CTR",txc),("TXN_X_CTR",txn)]:
    df.to_sql(nom, conn, if_exists="replace", index=False)

print("Données chargées dans Python et SQLite.")

## Module 1 — CTEs (WITH SQL)
### 1.1 CTE en SQL

🔎 **En clair :** une CTE (`WITH ... AS`) est juste un **nom** qu'on donne à un résultat
de requête intermédiaire, pour le réutiliser ensuite dans la requête finale — comme une
variable qui contiendrait un tableau. En pandas, on fait exactement la même chose avec une
variable normale (ex : `ctr_actifs = ctr[...]`, puis on réutilise `ctr_actifs` plus loin).

In [ ]:
# ── CTEs — Common Table Expressions ───────────────────────────────────────
# Une CTE nomme une sous-requête pour la réutiliser dans la requête principale
# Syntaxe SQL standard : SQLite, PostgreSQL, Vertica...

sql_cte = """
    -- CTE 1 : contrats actifs avec solde
    WITH ctr_actifs AS (
        SELECT IDT_AC, SLD_CTR, COD_DEV
        FROM CTR
        WHERE COD_ECV_CTR = '1'
    ),
    -- CTE 2 : nombre de transactions par compte
    nb_txn AS (
        SELECT IDT_AC, COUNT(*) AS NB_TXN, SUM(MNT_MVT) AS MNT_TOT
        FROM TXN_X_CTR
        GROUP BY IDT_AC
    )
    -- Requête finale : joindre les deux CTEs
    SELECT a.IDT_AC,
           a.SLD_CTR,
           COALESCE(t.NB_TXN, 0)    AS NB_TXN,
           COALESCE(t.MNT_TOT, 0.0) AS MNT_TOT
    FROM ctr_actifs a
    LEFT JOIN nb_txn t ON a.IDT_AC = t.IDT_AC
    ORDER BY NB_TXN DESC
    LIMIT 10
"""
# Exécuter la requête SQL et récupérer le résultat dans un DataFrame
resultat_cte = pd.read_sql(sql_cte, conn)
print(resultat_cte.round(2))

### 1.2 Équivalent Pandas — CTEs avec variables intermédiaires

In [ ]:
# ── CTEs reproduites en Pandas ────────────────────────────────────────────
# Chaque CTE SQL = une variable DataFrame intermédiaire nommée
# Cette approche est PLUS lisible qu'une longue chaîne de méthodes

# CTE 1 : ctr_actifs — contrats ouverts
ctr_actifs = (ctr[ctr["COD_ECV_CTR"] == "1"]          # filtre statut ouvert
                 [["IDT_AC", "SLD_CTR", "COD_DEV"]]    # garder 3 colonnes
                 .copy())                               # .copy() évite les warnings

# CTE 2 : nb_txn — agrégation transactions
nb_txn = (txn.groupby("IDT_AC")
             .agg(NB_TXN = ("NUM_ORD_MVT_CPB", "count"),
                  MNT_TOT = ("MNT_MVT", "sum"))
             .reset_index())

# Requête finale : LEFT JOIN + remplir les NaN + tri
resultat = (pd.merge(ctr_actifs, nb_txn, on="IDT_AC", how="left")
              .fillna({"NB_TXN": 0, "MNT_TOT": 0})
              .sort_values("NB_TXN", ascending=False)
              .head(10))
print(resultat)

### ✏️ Exercice 1 — Top clients par volume

In [ ]:
# ── EXERCICE 1 ─────────────────────────────────────────────────────────────
# Construire la vue "top clients par volume de transactions"
# Étapes :
# 1. CTE1 : agréger TXN_X_CTR par IDT_AC (total montants, nb transactions)
# 2. CTE2 : joindre avec TIE_X_CTR pour avoir IDT_PI
# 3. Joindre avec TIE pour avoir COD_LNG_CTR
# 4. Grouper par COD_LNG_CTR : N clients, MNT_TOT moyen
# 5. Trier par MNT_TOT décroissant

# --- votre code ---

### ✅ Correction Exercice 1

In [ ]:
# ── CORRECTION EXERCICE 1 ─────────────────────────────────────────────────
# CTE1 : transactions agrégées par compte
cte1 = txn.groupby("IDT_AC").agg(
    NB_TXN  = ("NUM_ORD_MVT_CPB", "count"),
    MNT_TOT = ("MNT_MVT", "sum")
).reset_index()

# CTE2 : ajouter IDT_PI (lien compte-client)
cte2 = pd.merge(cte1, txc, on="IDT_AC", how="left")

# CTE3 : ajouter la langue du client
cte3 = pd.merge(cte2, tie[["IDT_PI","COD_LNG_CTR"]], on="IDT_PI", how="left")

# Analyse par langue
par_langue = cte3.groupby("COD_LNG_CTR").agg(
    N_CLIENTS  = ("IDT_PI", "nunique"),
    MNT_MOYEN  = ("MNT_TOT", "mean"),
    NB_TXN_MOY = ("NB_TXN", "mean")
).reset_index()
print(par_langue.round(2))

## Module 2 — OVER PARTITION BY / .transform()
### 2.1 Concept

🔎 **En clair :** `PARTITION BY` calcule un résultat **par groupe**, mais **garde
toutes les lignes** (contrairement à `groupby()` seul, qui résume en une ligne par groupe).
`.transform()` fait exactement ça en pandas : autant de lignes en sortie qu'en entrée.

In [ ]:
# ── .transform() — OVER PARTITION BY en Pandas ───────────────────────────
# SQL : SELECT *, SUM(MNT_MVT) OVER (PARTITION BY IDT_AC) AS TOTAL_COMPTE
# Python : .groupby("IDT_AC")["MNT_MVT"].transform("sum")
#
# Différence GROUP BY vs PARTITION BY :
# - GROUP BY : retourne UNE ligne par groupe (agrégation)
# - PARTITION BY : retourne TOUTES les lignes + résultat du groupe sur chaque ligne

# Calculer le total des transactions de chaque compte — sur CHAQUE ligne
txn["TOTAL_COMPTE"] = txn.groupby("IDT_AC")["MNT_MVT"].transform("sum")

# Calculer le nombre de transactions de chaque compte — sur chaque ligne
txn["NB_TXN_COMPTE"] = txn.groupby("IDT_AC")["MNT_MVT"].transform("count")

# Calculer le % que représente chaque transaction dans son compte
txn["PCT_SUR_COMPTE"] = (txn["MNT_MVT"] / txn["TOTAL_COMPTE"] * 100).round(2)

# Vérification
print(txn[["IDT_AC","MNT_MVT","TOTAL_COMPTE","NB_TXN_COMPTE","PCT_SUR_COMPTE"]].head(8))

### ✏️ Exercice 2 — Poids par contrat

In [ ]:
# ── EXERCICE 2 ─────────────────────────────────────────────────────────────
# Joindre CTR avec TIE_X_CTR pour avoir IDT_PI
# Calculer :
# 1. TOTAL_CLIENT = somme SLD_CTR par IDT_PI (.transform("sum"))
# 2. PCT_CONTRAT  = SLD_CTR / TOTAL_CLIENT * 100
# 3. RANG_SOLDE   = rang par SLD_CTR décroissant dans chaque groupe IDT_PI
#                   (utiliser .transform(lambda x: x.rank(ascending=False, method="dense")))

# --- votre code ---

### ✅ Correction Exercice 2

In [ ]:
# ── CORRECTION EXERCICE 2 ─────────────────────────────────────────────────
# Jointure CTR + TIE_X_CTR
vue = pd.merge(ctr, txc, on="IDT_AC", how="left")

# 1. Total des soldes par client
vue["TOTAL_CLIENT"] = vue.groupby("IDT_PI")["SLD_CTR"].transform("sum")

# 2. % de ce contrat dans le portefeuille du client
vue["PCT_CONTRAT"] = (vue["SLD_CTR"] / vue["TOTAL_CLIENT"] * 100).round(2)

# 3. Rang dans le portefeuille du client (1 = plus gros solde)
# method="dense" + Int64 (nullable) : gère les soldes manquants sans planter
vue["RANG_SOLDE"] = vue.groupby("IDT_PI")["SLD_CTR"].transform(
    lambda x: x.rank(ascending=False, method="dense")
).astype("Int64")

# Nombre de contrats par client, puis clients possédant plusieurs contrats
vue["NB_CTR"] = vue.groupby("IDT_PI")["IDT_AC"].transform("count")
multi = vue[vue["NB_CTR"] > 1]
print(vue[["IDT_PI","IDT_AC","SLD_CTR","TOTAL_CLIENT","PCT_CONTRAT","RANG_SOLDE"]].head(10))

## Module 3 — ROW_NUMBER, LAG, LEAD
### 3.1 .cumcount() et .shift()

🔎 **En clair :** `ROW_NUMBER()` numérote les lignes (1, 2, 3...) à l'intérieur d'un
groupe. `LAG()` va chercher la valeur de la ligne **précédente**. En pandas :
`.cumcount()` et `.shift()` font respectivement la même chose.

In [ ]:
# ── ROW_NUMBER() — numérotation dans un groupe ────────────────────────────
# SQL : ROW_NUMBER() OVER (PARTITION BY IDT_AC ORDER BY DAT_MVT)
# Python : .cumcount() + 1 après sort + groupby

# Trier d'abord (IMPORTANT : le tri doit précéder le cumcount)
txn["DAT_MVT"] = pd.to_datetime(txn["DAT_MVT"])  # parser la date
txn_tri = txn.sort_values(["IDT_AC", "DAT_MVT"]).copy()

# cumcount() commence à 0 → ajouter 1 pour commencer à 1
txn_tri["RANG_TXN"] = txn_tri.groupby("IDT_AC").cumcount() + 1

# Identifier la PREMIÈRE transaction de chaque compte
premiere = txn_tri[txn_tri["RANG_TXN"] == 1]
print(f"Nombre de premiers comptes : {len(premiere)}")
print(premiere[["IDT_AC","DAT_MVT","MNT_MVT","RANG_TXN"]].head(5))

In [ ]:
# ── LAG et LEAD — décalage temporel ──────────────────────────────────────
# Python : .shift(1)  → décaler vers le bas de 1 ligne

# LAG : montant de la transaction précédente dans le même compte
txn_tri["MNT_PRECEDENT"] = (txn_tri.groupby("IDT_AC")["MNT_MVT"]
                                    .shift(1))     # shift(1) = ligne précédente

# Variation absolue par rapport à la transaction précédente
txn_tri["VARIATION"] = txn_tri["MNT_MVT"] - txn_tri["MNT_PRECEDENT"]

# LEAD : montant de la prochaine transaction
txn_tri["MNT_SUIVANT"] = (txn_tri.groupby("IDT_AC")["MNT_MVT"]
                                  .shift(-1))    # shift(-1) = ligne suivante

# Vérification
print(txn_tri[["IDT_AC","DAT_MVT","MNT_MVT","MNT_PRECEDENT","VARIATION"]].head(8))

### ✏️ Exercice 3 — Transactions anormales

In [ ]:
# ── EXERCICE 3 ─────────────────────────────────────────────────────────────
# 1. Trier TXN_X_CTR par IDT_AC et DAT_MVT
# 2. Calculer RANG_TXN avec cumcount()
# 3. Calculer MNT_PRECEDENT avec .shift(1) par groupe IDT_AC
# 4. Calculer RATIO = abs(MNT_MVT) / abs(MNT_PRECEDENT) — multiplication par rapport au précédent
# 5. Créer ALERTE = "OUI" si RATIO > 3 (montant plus que triplé)
# 6. Afficher le count des alertes

# --- votre code ---

### ✅ Correction Exercice 3

In [ ]:
# ── CORRECTION EXERCICE 3 ─────────────────────────────────────────────────
# 1-2. Trier + numéroter
txn["DAT_MVT"] = pd.to_datetime(txn["DAT_MVT"])
txn_a = txn.sort_values(["IDT_AC", "DAT_MVT"]).copy()
txn_a["RANG"] = txn_a.groupby("IDT_AC").cumcount() + 1

# 3. LAG
txn_a["MNT_PREC"] = txn_a.groupby("IDT_AC")["MNT_MVT"].shift(1)

# 4. Ratio (valeurs absolues pour éviter les divisions par zéro)
txn_a["RATIO"] = (txn_a["MNT_MVT"].abs() / txn_a["MNT_PREC"].abs()).round(2)

# 5. Alerte
txn_a["ALERTE"] = np.where(txn_a["RATIO"] > 3, "OUI", "NON")

# 6. Résultat
print("Alertes :")
print(txn_a["ALERTE"].value_counts())
print("\nExemples d'alertes :")
print(txn_a[txn_a["ALERTE"]=="OUI"][["IDT_AC","DAT_MVT","MNT_MVT","MNT_PREC","RATIO"]].head(5))

## Module 4 — Dates et Composantes
### 4.1 Parsing et extraction

In [ ]:
# ── Formats de dates Beobank ──────────────────────────────────────────────
# Deux formats coexistent dans les fichiers Beobank :
# 1. YYYY-MM-DD   → CTR, TIE, TXN_X_CTR  (format standard ISO)
# 2. DDMONYYYY    → TIE_ADR (ex: "24NOV2025") — format historique

# Format standard — pd.to_datetime() fonctionne directement
ctr["DAT_OUV_CTR"] = pd.to_datetime(ctr["DAT_OUV_CTR"])

# Format DDMONYYYY — format="mixed" détecte automatiquement
tie_adr["DAT_MAJ_ADR"] = pd.to_datetime(tie_adr["DAT_MAJ_ADR"], format="mixed")

# Extraction des composantes de date via l'accesseur .dt
ctr["ANNEE"]    = ctr["DAT_OUV_CTR"].dt.year
ctr["MOIS"]     = ctr["DAT_OUV_CTR"].dt.month
ctr["JOUR"]     = ctr["DAT_OUV_CTR"].dt.day
ctr["TRIMESTRE"]= ctr["DAT_OUV_CTR"].dt.quarter       # 1, 2, 3 ou 4
ctr["JOUR_SEM"] = ctr["DAT_OUV_CTR"].dt.day_name()    # "Monday", "Tuesday"...

# Ancienneté du contrat en jours
ctr["ANCIENNETE_J"] = (pd.Timestamp("today") - ctr["DAT_OUV_CTR"]).dt.days

print(ctr[["IDT_AC","DAT_OUV_CTR","ANNEE","MOIS","TRIMESTRE","ANCIENNETE_J"]].head(5))

## Module 5 — Fenêtre glissante 13 mois
### 5.1 rolling() et DateOffset

🔎 **En clair :** une **fenêtre glissante** (`rolling`) regarde toujours les N
dernières lignes et calcule une statistique dessus (ex : moyenne des 3 derniers mois),
puis avance d'une ligne à la fois — comme une moyenne mobile.

In [ ]:
# ── Séries temporelles — agréger par mois ─────────────────────────────────
# Préparer les données : parser la date et créer une période mensuelle
txn["DAT_MVT"] = pd.to_datetime(txn["DAT_MVT"])

# .dt.to_period("M") → convertit la date en période mensuelle (ex: "2024-01")
txn["ANNEE_MOIS"] = txn["DAT_MVT"].dt.to_period("M")

# Agréger par mois — série temporelle du volume
mensuel = (txn.groupby("ANNEE_MOIS")
              .agg(NB_TXN   = ("MNT_MVT", "count"),
                   MNT_TOTAL = ("MNT_MVT", "sum"))
              .reset_index()
              .sort_values("ANNEE_MOIS"))

print(f"Période couverte : {mensuel['ANNEE_MOIS'].min()} → {mensuel['ANNEE_MOIS'].max()}")
print(mensuel.head(6))

In [ ]:
# ── rolling() — fenêtres glissantes ──────────────────────────────────────
# rolling(N) : calculer une statistique sur les N dernières lignes

# Moyenne mobile 3 mois — lisse les fluctuations saisonnières
mensuel["MNT_MOY_3M"] = (mensuel["MNT_TOTAL"]
                          .rolling(window=3, min_periods=1)  # min_periods=1 : calculer même avec < 3 valeurs
                          .mean()
                          .round(2))

# Cumul glissant 13 mois — rapport "glissant annuel"
mensuel["MNT_ROLLING_13M"] = (mensuel["MNT_TOTAL"]
                               .rolling(window=13, min_periods=1)
                               .sum()
                               .round(2))

# Variation mensuelle en % (pct_change = ((mois_n / mois_n-1) - 1) * 100)
mensuel["VAR_PCT"] = mensuel["MNT_TOTAL"].pct_change() * 100

print(mensuel[["ANNEE_MOIS","MNT_TOTAL","MNT_MOY_3M","VAR_PCT"]].head(8).round(2))

### ✏️ Exercice 4 — Évolution mensuelle

In [ ]:
# ── EXERCICE 4 ─────────────────────────────────────────────────────────────
# 1. Parser DAT_MVT dans TXN_X_CTR
# 2. Agréger par mois : nb transactions + montant total
# 3. Moyenne mobile 3 mois (rolling(3).mean())
# 4. Variation mensuelle en % (pct_change())
# 5. Quel mois a le plus de transactions ? (.idxmax())

# --- votre code ---

### ✅ Correction Exercice 4

In [ ]:
# ── CORRECTION EXERCICE 4 ─────────────────────────────────────────────────
# 1-2. Parser + agréger
txn["DAT_MVT"] = pd.to_datetime(txn["DAT_MVT"])
txn["PERIODE"]  = txn["DAT_MVT"].dt.to_period("M")

stats_mois = txn.groupby("PERIODE").agg(
    NB   = ("MNT_MVT", "count"),
    TOT  = ("MNT_MVT", "sum")
).reset_index().sort_values("PERIODE")

# 3. Moyenne mobile
stats_mois["MOY_3M"] = stats_mois["NB"].rolling(3, min_periods=1).mean().round(1)

# 4. Variation %
stats_mois["VAR_PCT"] = (stats_mois["TOT"].pct_change() * 100).round(1)

# 5. Mois record
idx_max = stats_mois["NB"].idxmax()
print(f"Mois le plus actif : {stats_mois.loc[idx_max,'PERIODE']} ({stats_mois.loc[idx_max,'NB']} txn)")
print(stats_mois.head(8))

## Module 6 — Matplotlib
### 6.1 Graphiques bar et barh

🔎 **En clair :** un graphique matplotlib se construit toujours pareil :
1) créer une figure (`fig, ax = plt.subplots()`), 2) dessiner dessus (`ax.bar`, `ax.plot`...),
3) ajouter titres et légendes, 4) afficher avec `plt.show()`.

In [ ]:
# ── Graphiques Matplotlib — structure de base ─────────────────────────────
# fig, ax = plt.subplots(figsize=(largeur, hauteur)) — toujours commencer par ça

import os; os.makedirs("../output", exist_ok=True)

# ── 1. BARRES VERTICALES — distribution des statuts ───────────────────────
freq_statuts = ctr["COD_ECV_CTR"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(freq_statuts.index,          # axe X : codes statuts
       freq_statuts.values,          # axe Y : nombre de contrats
       color="#0E8C86",              # couleur turquoise Beobank
       edgecolor="white",            # bordure blanche entre barres
       linewidth=0.8)
ax.set_title("Contrats par statut — Beobank", fontsize=14, fontweight="bold")
ax.set_xlabel("Code statut")
ax.set_ylabel("Nombre de contrats")
ax.grid(axis="y", alpha=0.3)        # grille horizontale discrète
plt.tight_layout()
plt.savefig("../output/barres_statuts.png", dpi=150)
plt.show()

In [ ]:
# ── 2. BARRES HORIZONTALES — plus lisibles pour les libellés longs ─────────
lib_statuts = {"1":"Ouvert","2":"En attente","3":"Suspendu",
               "4":"Clôturé","5":"En résiliation","6":"Résilié"}
freq_lib = freq_statuts.copy()
freq_lib.index = [lib_statuts.get(c, c) for c in freq_lib.index]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(freq_lib.index, freq_lib.values,       # barh = barres horizontales
        color="#0B2D48",                         # bleu marine Beobank
        edgecolor="white")
ax.set_title("Répartition des statuts de contrat", fontsize=13, fontweight="bold")
ax.set_xlabel("Nombre de contrats")

# Ajouter les valeurs sur les barres
for i, v in enumerate(freq_lib.values):
    ax.text(v + 0.5, i, str(v), va="center", fontsize=10)

plt.tight_layout()
plt.savefig("../output/barres_h_statuts.png", dpi=150)
plt.show()

In [ ]:
# ── 3. COURBE AVEC DOUBLE AXE — twinx() ─────────────────────────────────
# Afficher montant (courbe) et nb transactions (barres) sur le même graphique

mois_labels = mensuel["ANNEE_MOIS"].astype(str)

fig, ax1 = plt.subplots(figsize=(12, 5))

# Axe gauche (principal) — montant total
ax1.plot(mois_labels, mensuel["MNT_TOTAL"],
         color="#0E8C86", marker="o", linewidth=2.5, label="Montant total (EUR)")
ax1.set_ylabel("Montant total (EUR)", color="#0E8C86", fontsize=11)
ax1.tick_params(axis="y", labelcolor="#0E8C86")

# Axe droit (secondaire) — nb transactions
ax2 = ax1.twinx()                    # partage l'axe X, crée un axe Y indépendant
ax2.bar(mois_labels, mensuel["NB_TXN"],
        alpha=0.3, color="#0B2D48", label="Nb transactions")
ax2.set_ylabel("Nombre de transactions", color="#0B2D48", fontsize=11)
ax2.tick_params(axis="y", labelcolor="#0B2D48")

ax1.set_title("Transactions Beobank — Montant et Volume mensuel", fontsize=13)
plt.xticks(rotation=45, ha="right")

# Légende combinée des deux axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.savefig("../output/courbe_twinx.png", dpi=150)
plt.show()

In [ ]:
# ── 4. CAMEMBERT — répartition FR vs NL ─────────────────────────────────
lng = tie["COD_LNG_CTR"].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(lng.values,
       labels=lng.index,
       autopct="%1.1f%%",              # afficher le % dans chaque part
       colors=["#0E8C86", "#0B2D48"], # turquoise et bleu marine
       startangle=90,                  # commencer à 12h
       wedgeprops={"edgecolor": "white", "linewidth": 2})
ax.set_title("Clients Beobank par langue du contrat", fontsize=13)
plt.tight_layout()
plt.savefig("../output/camembert_langue.png", dpi=150)
plt.show()

In [ ]:
# ── 5. DASHBOARD GRIDSPEC — 4 graphiques en 1 ────────────────────────────
# GridSpec permet de composer des layouts complexes
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(14, 9))
fig.suptitle("Dashboard Portefeuille Beobank", fontsize=16, fontweight="bold", y=1.01)

# Créer une grille 2 lignes × 2 colonnes
gs = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.35)
ax1 = fig.add_subplot(gs[0, 0])  # haut gauche
ax2 = fig.add_subplot(gs[0, 1])  # haut droite
ax3 = fig.add_subplot(gs[1, 0])  # bas gauche
ax4 = fig.add_subplot(gs[1, 1])  # bas droite

# Graphique 1 : nb contrats par statut
freq = ctr["COD_ECV_CTR"].value_counts().sort_index()
ax1.bar(freq.index, freq.values, color="#0E8C86")
ax1.set_title("Contrats par statut")
ax1.set_xlabel("Statut"); ax1.set_ylabel("Nb")

# Graphique 2 : camembert FR/NL
lng = tie["COD_LNG_CTR"].value_counts()
ax2.pie(lng.values, labels=lng.index, autopct="%1.1f%%",
        colors=["#0E8C86","#0B2D48"], startangle=90)
ax2.set_title("Répartition linguistique")

# Graphique 3 : évolution mensuelle nb transactions
ax3.plot(range(len(mensuel)), mensuel["NB_TXN"], color="#0E8C86", marker=".")
ax3.set_title("Évolution mensuelle (nb txn)")
ax3.set_xlabel("Mois"); ax3.set_ylabel("Nb transactions")

# Graphique 4 : distribution des segments de solde
ctr["SEGMENT"] = np.select(
    [ctr["SLD_CTR"]<0, ctr["SLD_CTR"]<5000, ctr["SLD_CTR"]<50000],
    ["Critique","Faible","Moyen"], default="Élevé")
seg = ctr["SEGMENT"].value_counts()
ax4.barh(seg.index, seg.values, color="#0B2D48")
ax4.set_title("Segments de solde")

plt.tight_layout()
plt.savefig("../output/dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Dashboard sauvegardé : ../output/dashboard.png")

### ✏️ Exercice 5 — Tableau de bord

In [ ]:
# ── EXERCICE 5 ─────────────────────────────────────────────────────────────
# Créez un dashboard GridSpec 2x2 :
# ax1 : barres horizontales — nb contrats par statut (libellés complets)
# ax2 : camembert — répartition FR vs NL
# ax3 : courbe — évolution mensuelle des transactions
# ax4 : barres — distribution des SEGMENTS de solde
# Sauvegarder en PNG

# --- votre code ---

## Module 7 — resample(), pct_change() et rank()
### 7.1 resample — rééchantillonnage temporel

In [ ]:
# ── resample() — agréger une série temporelle par période ────────────────
# Équivalent d'un groupby() mais spécialisé pour les dates : plus direct
txn_indexe = txn.set_index("DAT_MVT")                 # resample() exige une date en index

mensuel_resample = txn_indexe["MNT_MVT"].resample("ME").agg(["sum", "count"])   # "ME" = fin de mois
print(mensuel_resample.head(6))

### 7.2 pct_change — variations en pourcentage

In [ ]:
# ── pct_change() — variation en % entre lignes consécutives ──────────────
mensuel_resample["VARIATION_PCT"] = mensuel_resample["sum"].pct_change() * 100
print(mensuel_resample[["sum", "VARIATION_PCT"]].round(2).head(8))

# periods=N : comparer avec N lignes en arrière (ex : variation sur 3 mois)
mensuel_resample["VARIATION_3M"] = mensuel_resample["sum"].pct_change(periods=3) * 100

### 7.3 rank — classement par groupe

In [ ]:
# ── rank() — classement des lignes (équivalent d'un RANK SQL) ────────────
ctr["RANG_SOLDE"] = ctr["SLD_CTR"].rank(ascending=False, method="dense")   # 1 = solde le plus élevé

top5 = ctr.sort_values("RANG_SOLDE").head(5)
print(top5[["IDT_AC", "SLD_CTR", "RANG_SOLDE"]])

### ✏️ Exercice 6 — Top 3 transactions du mois

In [ ]:
# ── EXERCICE 6 ─────────────────────────────────────────────────────────────
# TODO 1 : ajouter ANNEE_MOIS sur txn (voir Module 5)
# TODO 2 : calculer un rang par mois du montant de chaque transaction :
#          txn.groupby("ANNEE_MOIS")["MNT_MVT"].rank(ascending=False)
# TODO 3 : garder seulement les lignes où le rang <= 3 (top 3 du mois)
# TODO 4 : afficher le résultat trié par mois puis par rang

### ✅ Correction Exercice 6

In [ ]:
# ── CORRECTION EXERCICE 6 ─────────────────────────────────────────────────
txn["ANNEE_MOIS"] = txn["DAT_MVT"].dt.to_period("M")

# Rang du montant DANS chaque mois (groupby().rank() = classement par groupe)
txn["RANG_MOIS"] = txn.groupby("ANNEE_MOIS")["MNT_MVT"].rank(ascending=False, method="first")

top3_par_mois = txn[txn["RANG_MOIS"] <= 3].sort_values(["ANNEE_MOIS", "RANG_MOIS"])
print(top3_par_mois[["ANNEE_MOIS", "IDT_AC", "MNT_MVT", "RANG_MOIS"]].head(12))

## Module 8 — Écrire dans la base de données
### 8.1 Créer une table (CREATE TABLE)

In [ ]:
# ── CREATE TABLE — créer une nouvelle table en SQL ────────────────────────
# On écrit le SQL directement, comme dans n'importe quel outil SQL.

conn.execute("DROP TABLE IF EXISTS notes_clients")   # repartir propre si on relance la cellule

conn.execute("""
    CREATE TABLE notes_clients (
        IDT_AC      TEXT,
        NOTE        INTEGER,
        COMMENTAIRE TEXT
    )
""")
print("Table notes_clients créée.")

### 8.2 Ajouter des lignes (INSERT INTO)

In [ ]:
# ── INSERT INTO — ajouter des lignes ──────────────────────────────────────
conn.execute(
    "INSERT INTO notes_clients (IDT_AC, NOTE, COMMENTAIRE) VALUES (?, ?, ?)",
    ("AC00001", 8, "Bon client")
)

# executemany() : insérer plusieurs lignes d'un coup depuis une liste de tuples
nouvelles_lignes = [
    ("AC00002", 5, "Client moyen"),
    ("AC00003", 9, "Excellent client"),
]
conn.executemany(
    "INSERT INTO notes_clients (IDT_AC, NOTE, COMMENTAIRE) VALUES (?, ?, ?)",
    nouvelles_lignes
)
conn.commit()   # valider les changements dans la base

# Vérifier avec un SELECT
print(pd.read_sql("SELECT * FROM notes_clients", conn))

### 8.3 Modifier des lignes (UPDATE)

In [ ]:
# ── UPDATE — modifier des lignes existantes ────────────────────────────────
conn.execute(
    "UPDATE notes_clients SET NOTE = 10 WHERE IDT_AC = ?",
    ("AC00001",)
)
conn.commit()

print(pd.read_sql("SELECT * FROM notes_clients WHERE IDT_AC = 'AC00001'", conn))

### ✏️ Exercice 7 — Table de suivi

In [ ]:
# ── EXERCICE ────────────────────────────────────────────────────────────────
# TODO 1 : créer une table suivi_contact (IDT_AC TEXT, DATE_CONTACT TEXT, STATUT TEXT)
# TODO 2 : insérer 2 lignes avec executemany()
# TODO 3 : mettre à jour le STATUT d'une ligne avec UPDATE
# TODO 4 : vérifier avec un SELECT * FROM suivi_contact

### ✅ Correction Exercice 7

In [ ]:
# ── CORRECTION ─────────────────────────────────────────────────────────────
conn.execute("DROP TABLE IF EXISTS suivi_contact")
conn.execute("""
    CREATE TABLE suivi_contact (
        IDT_AC       TEXT,
        DATE_CONTACT TEXT,
        STATUT       TEXT
    )
""")

conn.executemany(
    "INSERT INTO suivi_contact (IDT_AC, DATE_CONTACT, STATUT) VALUES (?, ?, ?)",
    [("AC00001", "2026-01-10", "En attente"),
     ("AC00002", "2026-01-11", "En attente")]
)
conn.commit()

conn.execute("UPDATE suivi_contact SET STATUT = 'Traité' WHERE IDT_AC = 'AC00001'")
conn.commit()

print(pd.read_sql("SELECT * FROM suivi_contact", conn))

## Module 9 — Pipeline complet 5 étapes
### Pipeline Beobank FR vs NL

In [ ]:
# ── PIPELINE COMPLET — du chargement à la visualisation ──────────────────

# ── ÉTAPE 1 : IMPORT ─────────────────────────────────────────────────────
DATA   = Path("../data")
PARAMS = dict(sep=";", na_values=".", encoding="utf-8")
ctr_p = pd.read_csv(DATA/"CTR.csv", **PARAMS)
txc_p = pd.read_csv(DATA/"TIE_X_CTR.csv", **PARAMS)
tie_p = pd.read_csv(DATA/"TIE.csv", **PARAMS)
print("Étape 1 — Import : OK")

# ── ÉTAPE 2 : NETTOYAGE ──────────────────────────────────────────────────
ctr_p["DAT_OUV_CTR"] = pd.to_datetime(ctr_p["DAT_OUV_CTR"])
ctr_p = ctr_p.dropna(subset=["SLD_CTR"])   # supprimer les lignes sans solde
print(f"Étape 2 — Nettoyage : {len(ctr_p)} contrats avec solde valide")

# ── ÉTAPE 3 : TRANSFORMATION ─────────────────────────────────────────────
# Jointure pour avoir la langue du client
vue = pd.merge(pd.merge(ctr_p, txc_p, on="IDT_AC", how="left"),
               tie_p[["IDT_PI","COD_LNG_CTR","COD_TYP_TIE"]], on="IDT_PI", how="left")

# Segmentation
cond = [vue["SLD_CTR"]<0, vue["SLD_CTR"]<5000, vue["SLD_CTR"]<50000]
vue["SEGMENT"] = np.select(cond, ["Critique","Faible","Moyen"], default="Élevé")
print(f"Étape 3 — Transformation : {vue.shape}")

# ── ÉTAPE 4 : ANALYSE ────────────────────────────────────────────────────
rapport_lng = vue.groupby("COD_LNG_CTR").agg(
    N_CTR  = ("IDT_AC",  "count"),
    MOY    = ("SLD_CTR", "mean"),
    TOTAL  = ("SLD_CTR", "sum"),
    PCT_POS = ("SLD_CTR", lambda x: (x>0).mean()*100)
).reset_index()
print("Étape 4 — Analyse :")
print(rapport_lng.round(2))

# ── ÉTAPE 5 : EXPORT + VISUALISATION ─────────────────────────────────────
rapport_lng.to_csv("../output/rapport_fr_nl.csv", sep=";", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(rapport_lng["COD_LNG_CTR"], rapport_lng["N_CTR"],
            color=["#0E8C86","#0B2D48"])
axes[0].set_title("Nb contrats FR vs NL")

axes[1].bar(rapport_lng["COD_LNG_CTR"], rapport_lng["MOY"],
            color=["#0E8C86","#0B2D48"])
axes[1].set_title("Solde moyen FR vs NL")

plt.tight_layout()
plt.savefig("../output/fr_vs_nl.png", dpi=150)
plt.show()
print("Étape 5 — Export et visualisation : OK")

## Exercice Final — Rapport complet Jour 3

In [ ]:
# ── EXERCICE FINAL ─────────────────────────────────────────────────────────
# Produire un rapport analytique complet avec :
#
# 1. CTE/variables intermédiaires :
#    - ctr_actifs : contrats ouverts (statut "1")
#    - txn_par_compte : NB_TXN et MNT_TOT par IDT_AC
# 2. Joindre les deux + ajouter la langue via TIE_X_CTR + TIE
# 3. Calculer RANG_COMPTE (par MNT_TOT dans chaque groupe de langue)
#    → .groupby("COD_LNG_CTR")["MNT_TOT"].transform(lambda x: x.rank(ascending=False, method="dense"))
# 4. Créer une série temporelle mensuelle des transactions
# 5. Graphique double : courbe montant mensuel + camembert FR/NL
# 6. Exporter le rapport en CSV

# --- votre code ---